In [ ]:
# VLA : Vision Language Model
# 멀티모달

# YOLO : 정보억제, 경량화
# image -> YOLO -> text -> LLM -> response

In [1]:
# PinkWink/pinky_mujoco_menagerie/pinky_mujoco_menagerie/scripts/best.pt
!pip install ultralytics

In [ ]:
import cv2
from ultralytics import YOLO
from PIL import Image

# 이미지 테스트
img_path = "./mujoco_objects.png"
frame = cv2.resize(cv2.imread(img_path), (640, 480))
model = YOLO("best.pt")
results = model(source=frame, conf=0.5, verbose=True)


0: 480x640 1 can, 1 milk, 1 lemon, 143.0ms
Speed: 65.5ms preprocess, 143.0ms inference, 10.2ms postprocess per image at shape (1, 3, 480, 640)


In [5]:
for i, r in enumerate(results):
    im_bgr = r.plot()
    im_rgb = Image.fromarray(im_bgr[..., ::-1])
    r.show()

In [ ]:
# 실시간 테스트
import numpy
import mujoco as mj
from mujoco.glfw import glfw        # 시뮬레이터 (OpenGL기반)

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".")))

from utils.mouse_callbacks import MouseCallbacks            # 마우스 이벤트 (카메라 회전/이동/줌 등) 콜백
from utils.keyboard_callbacks import keyboard_callbacks     # 키보드 입력(로봇 제어 및 초기화) 콜백
from utils.scene_creator import SceneCreator        # MJCF xml 파일 조합 및 scene 빌드용 핼퍼

ImportError: cannot import name 'utils' from 'mujoco' (c:\Python312\Lib\site-packages\mujoco\__init__.py)

In [6]:
cwd = os.getcwd()
PROJECT_ROOT = os.path.dirname(cwd)
print(f"project root: {PROJECT_ROOT}")

assets_dir = os.path.join(PROJECT_ROOT, "assets")
base_env_path = os.path.join(assets_dir, "scenes", "floor_sky.xml")     # 환경 MJCF
robot_path = os.path.join(assets_dir, "robots", "pinky", "pinky.xml")   # 로봇 MJCF

# 배치할 오브젝트 정보
objects_to_spawn = [
    {"path": os.path.join(assets_dir, "objects", "can.xml"), "name":"can_1", "pos": "0.5 0.3 0.05"},
    {"path": os.path.join(assets_dir, "objects", "milk.xml"), "name": "milk_1", "pos":"0.5 0.0 0.05"},
    {"path": os.path.join(assets_dir, "objects", "lemon.xml"), "name": "lemon_1", "pos":"0.5 -0.3 0.05"}
]

# 각 MJCF들을 병합
scene_xml_string = SceneCreator.build_mjcf_scene(
    base_env_path = base_env_path,
    robot_path = robot_path,
    objects_to_spawn=objects_to_spawn,
    save_xml=False
)

NameError: name 'os' is not defined

In [ ]:
# XML문자열로부터 MuJoCo 모델(물리/구조 정보) 생성
model = mj.MjModel.from_xml_string(scene_xml_string)
# 모델에 대한 시뮬레이션 데이터(상태 변수, 센서값 등) 생성
data = mj.MjData(model)

# mujoco 카메라(메인, 로봇 시점), 옵션, 씬(각 시점의 3D 그래픽 정보) 객체 선언
main_cam = mj.MjvCamera()
robot_cam = mj.MjvCamera()
opt = mj.MjvOption()
scene_main = mj.MjvScene(model, maxgeom=10000)
scene_robot = mj.MjvScene(model, maxgeom=10000)

In [ ]:
# GLFW 초기화 및 윈도우 생성